# Gerar corpus_manifest_rename.yaml — Paula Femina

Baixa o PDF consolidado da RENAME 2024, calcula o hash SHA-256 e gera o
`corpus_manifest_rename.yaml` no mesmo formato usado pelo `corpus_manifest_proto.yaml`.

**Como usar:** Ambiente de execucao > Executar tudo. No final, baixe
`rename_output.zip` pelo icone de pasta na lateral.

In [ ]:
!pip -q install pyyaml requests

In [ ]:
import hashlib
import time
from pathlib import Path

import requests
import yaml

URL_RENAME = (
    "https://cdn.who.int/media/docs/default-source/essential-medicines/"
    "national-essential-medicines-lists-(neml)/paho_neml/brasil-neml-2024.pdf"
    "?sfvrsn=8af0afb7_1&download=true"
)

PASTA_SAIDA = Path("/content/RENAME_dir")
PASTA_SAIDA.mkdir(parents=True, exist_ok=True)

CAMINHO_PDF = PASTA_SAIDA / "rename_2024.pdf"

resposta = requests.get(URL_RENAME, timeout=60)
resposta.raise_for_status()
CAMINHO_PDF.write_bytes(resposta.content)
print(f"OK: {CAMINHO_PDF} ({len(resposta.content)/1024:.1f} KB)")

In [ ]:
def sha256_arquivo(caminho):
    digest = hashlib.sha256()
    with open(caminho, "rb") as f:
        for pedaco in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(pedaco)
    return digest.hexdigest()

hash_pdf = sha256_arquivo(CAMINHO_PDF)
tamanho_bytes = CAMINHO_PDF.stat().st_size

print("SHA-256:", hash_pdf)
print("Tamanho (bytes):", tamanho_bytes)

In [ ]:
hoje = time.strftime("%Y-%m-%d", time.localtime())
agora_utc = time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())

metadados_extras = [
    "versao_edicao",
    "versao_portaria",
    "status",
    "data_inicio",
    "data_fim",
]

manifest = {
    "scenario_id": "rename-medicamentos-essenciais",
    "version": hoje,
    "description": (
        "Relacao Nacional de Medicamentos Essenciais (RENAME) 2024 "
        "- Ministerio da Saude"
    ),
    "bases": [
        {
            "base_id": "rename_medicamentos",
            "natureza": "estruturada",
            "fonte": "Ministerio da Saude - RENAME 2024",
            "acesso": {
                "tipo": "http_pdf",
                "urls": [URL_RENAME],
            },
            "selecao_deterministica": True,
            "volume_alvo": "1 documento (edicao consolidada RENAME 2024)",
            "formato": "pdf",
            "licenca": "Dados abertos gov.br - atribuicao",
            "sensitivity_default": "publico",
            "campo_vigencia": "data_inicio",
            "chunking_sugerido": (
                "1 medicamento/apresentacao = 1 chunk tabular serializado "
                "a partir do PDF consolidado"
            ),
            "metadados_extras": metadados_extras,
        }
    ],
    "generated_by": "paula_femina.gerar_corpus_manifest_rename",
    "generated_at": agora_utc,
    "documentos": [
        {
            "scenario_id": "rename-medicamentos-essenciais",
            "manifest_version": hoje,
            "description": (
                "Relacao Nacional de Medicamentos Essenciais (RENAME) 2024 "
                "- Ministerio da Saude"
            ),
            "base_id": "rename_medicamentos",
            "natureza": "estruturada",
            "fonte": "Ministerio da Saude - RENAME 2024",
            "acesso": {
                "tipo": "http_pdf",
                "url_origem": (
                    "http://bvsms.saude.gov.br/bvs/publicacoes/"
                    "relacao_nacional_medicamentos_2024.pdf"
                ),
                "url_pdf_resolvida": URL_RENAME,
            },
            "selecao_deterministica": True,
            "volume_alvo": "1 documento (edicao consolidada RENAME 2024)",
            "formato": "pdf",
            "licenca": "Dados abertos gov.br - atribuicao",
            "sensitivity_default": "publico",
            "campo_vigencia": "data_inicio",
            "chunking_sugerido": (
                "1 medicamento/apresentacao = 1 chunk tabular serializado "
                "a partir do PDF consolidado"
            ),
            "metadados_extras": metadados_extras,
            "versao_edicao": "RENAME 2024",
            "versao_portaria": "Portaria GM/MS no 6.324, de 26/12/2024",
            "data_inicio": "2024-12-26",
            "data_fim": None,
            "status": "vigente",
            "confirmacao": "explicita (ficha catalografica do proprio PDF cita a Portaria)",
            "observacao": (
                "Link oficial (bvsms.saude.gov.br) retornou erro 503 persistente "
                "em testes de 21-24/09/2026. Usado espelho hospedado pela OMS/OPAS "
                "(cdn.who.int), mesmo conteudo, verificado por hash SHA-256. "
                "O painel 'Rename em Tempo Real' NAO foi usado como fonte, pois "
                "muda de conteudo e quebraria selecao_deterministica."
            ),
            "arquivo_pdf": "RENAME_dir/rename_2024.pdf",
            "sha256": hash_pdf,
            "tamanho_bytes": tamanho_bytes,
            "content_type": "application/pdf",
        }
    ],
    "falhas": [],
}

with open("/content/corpus_manifest_rename.yaml", "w", encoding="utf-8") as f:
    yaml.safe_dump(manifest, f, allow_unicode=True, sort_keys=False, default_flow_style=False)

print("corpus_manifest_rename.yaml gerado com sucesso!")
print()
with open("/content/corpus_manifest_rename.yaml", encoding="utf-8") as f:
    print(f.read())

In [ ]:
import shutil
import os

os.makedirs("/content/saida", exist_ok=True)
shutil.copy("/content/corpus_manifest_rename.yaml", "/content/saida/corpus_manifest_rename.yaml")
shutil.copytree("/content/RENAME_dir", "/content/saida/RENAME_dir")

shutil.make_archive("/content/rename_output", "zip", "/content/saida")
print("Pronto! Baixe rename_output.zip pelo painel de Arquivos (icone de pasta) a esquerda.")
print("Dentro dele: corpus_manifest_rename.yaml e RENAME_dir/rename_2024.pdf")